<div class="alert alert-block alert-success">  

# App 1: 模型推理演示 (Model Inference Demo)

**项目:** FIT5196 A1 (Extended Part)   
**模块:** App 1 - Demo  
**环境:** Local Python Environment (CPU or GPU)  
**目标:** 加载训练好的 PyTorch 模型权重 (`best_multimodal_model.pth`)，对来自 Google Maps 的真实评论数据（文本/图像）进行实时评分预测。

</div>

**数据准备:**
* 测试图片存储于本地 `test_images/` 目录。
* 需手动从 Google Maps 收集若干真实评论案例（包含好评、差评、无图、有图等场景）。

**测试场景：**
1.  **纯文本 (Text Only)**
2.  **单图多模态 (Text + Single Image)**
3.  **多图多模态 (Text + Multi Images)** - 采用平均概率融合策略。

**技术策略:**
* **模型重构:** 在内存中完全复现训练时的双塔结构（RoBERTa + Swin + LoRA）。
* **多模态推理:**
    * **纯文本:** 传入零张量占位，利用 mask 机制跳过视觉计算。
    * **多图融合:** 采用 **Bag-of-Images** 策略，对同一评论的多张图片分别推理，最后取概率均值作为最终结果。

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from transformers import AutoTokenizer, AutoModel, SwinModel
from peft import get_peft_model, LoraConfig, TaskType
from PIL import Image
import numpy as np
import os
import matplotlib.pyplot as plt
import torch.nn.functional as F

# 抑制警告
import warnings
warnings.filterwarnings("ignore")

# 设备检测
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Inference Device: {device}")

## 1. 重建模型架构 (Reconstruct Model Architecture)

为了加载权重，必须在内存中实例化与训练时结构完全一致的模型类。
**注意:** 此处的 `PatchedSwinModel` 和 `MultimodalClassifier` 定义必须与训练代码逐行一致，以确保权重键值匹配。

In [ ]:
# === 必须与训练代码保持完全一致 ===

class PatchedSwinModel(SwinModel):
    def forward(self, *args, **kwargs):
        ignore_keys = ['input_ids', 'attention_mask', 'token_type_ids', 'inputs_embeds']
        clean_kwargs = {k: v for k, v in kwargs.items() if k not in ignore_keys}
        return super().forward(*args, **clean_kwargs)

class MultimodalClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        
        # 1. Text Encoder (RoBERTa-Base)
        self.text_encoder = AutoModel.from_pretrained('roberta-base')
        peft_config_text = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION, 
            r=16, lora_alpha=32, lora_dropout=0.1,
            target_modules=["query", "value"] 
        )
        self.text_encoder = get_peft_model(self.text_encoder, peft_config_text)
        text_dim = self.text_encoder.config.hidden_size

        # 2. Image Encoder (Swin-Base)
        self.img_encoder = PatchedSwinModel.from_pretrained('microsoft/swin-base-patch4-window7-224')
        peft_config_img = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=16, lora_alpha=32, lora_dropout=0.1,
            target_modules=["query", "value"] 
        )
        self.img_encoder = get_peft_model(self.img_encoder, peft_config_img)
        img_dim = self.img_encoder.config.hidden_size
        
        # 3. Classifier
        self.fusion_dim = text_dim + img_dim
        self.classifier = nn.Sequential(
            nn.Linear(self.fusion_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, image, img_mask):
        # Text Forward
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_feat = text_out.last_hidden_state[:, 0, :] 

        # Image Forward (Lazy Execution)
        batch_size = input_ids.shape[0]
        hidden_dim = self.img_encoder.config.hidden_size 
        img_feat = torch.zeros((batch_size, hidden_dim), device=input_ids.device, dtype=text_feat.dtype)
        
        has_img_idx = torch.nonzero(img_mask).squeeze(1)
        if len(has_img_idx) > 0:
            valid_images = image[has_img_idx]
            valid_img_out = self.img_encoder(pixel_values=valid_images)
            valid_img_feat = valid_img_out.pooler_output
            img_feat[has_img_idx] = valid_img_feat

        # Fusion
        combined_feat = torch.cat((text_feat, img_feat), dim=1)
        logits = self.classifier(combined_feat)
        return logits

## 2. 加载权重与配置预处理 (Load Weights & Preprocessing)

* **权重加载:** 从本地加载 `best_multimodal_model.pth`。
* **预处理:** * 文本: 使用 Tokenizer 截断至 128 长度。
    * 图片: 定义标准的 Resize (224x224) 与 Normalize 流程。

In [ ]:
# 1. 实例化模型
model = MultimodalClassifier(num_classes=5)

# 2. 加载权重
model_path = 'best_multimodal_model.pth'
if os.path.exists(model_path):
    # map_location='cpu' 确保在无 GPU 的笔记本上也能运行演示
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    print(f"✅ 成功加载模型权重: {model_path}")
else:
    raise FileNotFoundError(f"❌ 未找到模型文件: {model_path}，请确保文件在当前目录下。")

model.to(device)
model.eval() # 切换评估模式

# 3. 初始化工具
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ 预处理管道就绪。")

## 3. 定义推理核心逻辑 (Inference Logic Definition)

定义通用的推理函数 `predict_review` 和可视化函数 `show_images`。
* **IMG_DIR:** 定义测试图片存储目录。
* **输入处理:** 将文本转为 Token ID，将图片转为 Tensor。若无图片，则生成全零 Tensor 并标记 `mask=0` (触发模型的惰性计算机制，跳过 Swin 计算)。
* **多图融合:** 对于包含 $N$ 张图片的评论，模型执行 $N$ 次前向传播 (Text + Image_i)，取 $N$ 次输出概率的平均值作为最终结果。

In [ ]:
# === 1. 定义图片目录 (原 Step 3 的内容移到这里) ===
IMG_DIR = 'test_images'

# === 2. 辅助函数 ===
def load_and_process_image(img_name):
    path = os.path.join(IMG_DIR, img_name)
    if not os.path.exists(path):
        print(f"⚠️ 图片不存在: {path} (将作为纯文本处理)")
        return torch.zeros((3, 224, 224)), False
    
    try:
        img = Image.open(path).convert('RGB')
        return inference_transforms(img), True
    except Exception as e:
        print(f"❌ 图片读取错误 {img_name}: {e}")
        return torch.zeros((3, 224, 224)), False

def show_images(img_files):
    if not img_files: return
    valid_files = [f for f in img_files if os.path.exists(os.path.join(IMG_DIR, f))]
    if not valid_files: return
    
    plt.figure(figsize=(10, 3))
    for i, img_name in enumerate(valid_files):
        path = os.path.join(IMG_DIR, img_name)
        img = Image.open(path)
        plt.subplot(1, len(valid_files), i+1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(img_name)
    plt.show()

# === 3. 核心推理函数 ===
def predict_review(case):
    text = case['text']
    img_files = case['images']
    
    # 文本处理
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True, 
        padding='max_length', max_length=128
    )
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)
    
    # 图像处理
    image_tensors = []
    if len(img_files) == 0:
        # 无图: 全零占位，Mask=0
        image_tensors.append((torch.zeros((3, 224, 224)), 0))
    else:
        # 有图: 正常处理，Mask=1
        for img_name in img_files:
            img_tensor, is_valid = load_and_process_image(img_name)
            mask_val = 1 if is_valid else 0
            image_tensors.append((img_tensor, mask_val))
    
    # 模型推理 (多图投票)
    all_logits = []
    with torch.no_grad():
        for img_tensor, mask_val in image_tensors:
            img_input = img_tensor.unsqueeze(0).to(device)
            mask_input = torch.tensor([mask_val]).to(device)
            logits = model(input_ids, attention_mask, img_input, mask_input)
            all_logits.append(logits)
    
    # 结果聚合
    stacked_logits = torch.stack(all_logits).mean(dim=0)
    probabilities = F.softmax(stacked_logits, dim=1).cpu().numpy()[0]
    predicted_class = np.argmax(probabilities) + 1 
    
    return predicted_class, probabilities

print("推理管道已就绪。")

## 4. 场景测试 1: 纯文本 (Text Only)

模拟用户仅提供文本评论的场景。
* **测试数据:** 构造一个只包含负面文本描述、无图片的字典。
* **预期行为:** 模型应仅依赖 RoBERTa 提取的文本特征进行判断。

In [ ]:
# === 场景 1: 纯文本差评 ===
case_text_only = {
    "id": "Case 1 (Negative Text Only)",
    "text": "The chicken was excessively oily and the batter was falling off. Not worth the wait.",
    "images": [], # 空列表
    "true_rating": 1
}

print(f"🔹 ID: {case_text_only['id']}")
print(f"📄 Text: \"{case_text_only['text']}\"")
print("🖼️ Images: None")

# 执行预测
pred, probs = predict_review(case_text_only)

print(f"\n⭐ True Rating: {case_text_only['true_rating']} Stars")
print(f"🤖 Predicted:   {pred} Stars")
print(f"📊 Confidence:  {probs[pred-1]:.4f}")
print(f"📈 Probabilities: {np.round(probs, 3)}")

## 5. 场景测试 2: 单图多模态 (Text + Single Image)

模拟最常见的多模态评论场景。
* **测试数据:** 包含一段好评文本和一张展示食物的图片。
* **预期行为:** 模型同时激活 RoBERTa 和 Swin Transformer，融合两者的特征向量进行预测。

In [ ]:
# === 场景 2: 文本 + 单张好评图 ===
# 请确保 'chicken_good.jpg' 存在于 test_images 文件夹中
case_single_img = {
    "id": "Case 2 (Positive with 1 Image)",
    "text": "Absolutely delicious! Huge portion size and very crispy. Best fried chicken in town.",
    "images": ["chicken_good.jpg"], 
    "true_rating": 5
}

print(f"🔹 ID: {case_single_img['id']}")
print(f"📄 Text: \"{case_single_img['text']}\"")

# 执行预测
pred, probs = predict_review(case_single_img)

# 展示结果
show_images(case_single_img['images'])
print(f"\n⭐ True Rating: {case_single_img['true_rating']} Stars")
print(f"🤖 Predicted:   {pred} Stars")
print(f"📊 Confidence:  {probs[pred-1]:.4f}")
print(f"📈 Probabilities: {np.round(probs, 3)}")

## 6. 场景测试 3: 多图多模态 (Text + Multi Images)

模拟用户上传多张图片的复杂场景。
* **测试数据:** 包含中性/混合评价的文本以及多张图片（如食物特写和店铺环境）。
* **预期行为:** 模型采用 **"Bag-of-Images"** 策略，综合多张图片的视觉信息进行平均加权。

In [ ]:
# === 场景 3: 文本 + 多张中评图 ===
# 请确保 'chicken_dry.jpg' 和 'store_front.jpg' 存在于 test_images 文件夹中
case_multi_img = {
    "id": "Case 3 (Neutral with 2 Images)",
    "text": "It was okay. The seasoning was good but the meat was a bit dry inside.",
    "images": ["chicken_dry.jpg", "store_front.jpg"], 
    "true_rating": 3
}

print(f"🔹 ID: {case_multi_img['id']}")
print(f"📄 Text: \"{case_multi_img['text']}\"")

# 执行预测
pred, probs = predict_review(case_multi_img)

# 展示结果
show_images(case_multi_img['images'])
print(f"\n⭐ True Rating: {case_multi_img['true_rating']} Stars")
print(f"🤖 Predicted:   {pred} Stars")
print(f"📊 Confidence:  {probs[pred-1]:.4f}")
print(f"📈 Probabilities: {np.round(probs, 3)}")